In [ ]:
# DOMAIN: finance
# SUBJECT_AREA: revenue
# SCHEDULE: 0 6 * * 1
# OWNER: analytics-team


## Revenue Analysis Pipeline
Tests Jupyter notebook lineage extraction with embedded SQL and PySpark.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = SparkSession.builder.appName("RevenueAnalysis").getOrCreate()

# Read source tables
revenue_df = spark.table("silver.daily_revenue")
products_df = spark.read.table("silver.products")
regions_df = spark.table("silver.regions")


In [ ]:
# SQL magic cell - embedded SQL in notebook
sql_result = spark.sql("""
    SELECT
        r.revenue_date,
        r.product_id,
        r.region_id,
        r.gross_amount,
        r.net_amount,
        r.discount_amount,
        p.product_name,
        p.product_category,
        rg.region_name,
        rg.country
    FROM silver.daily_revenue r
    JOIN silver.products p ON r.product_id = p.product_id
    JOIN silver.regions rg ON r.region_id = rg.region_id
    WHERE r.revenue_date >= "2024-01-01"
""")


In [ ]:
# Aggregate and write
monthly_summary = sql_result.groupBy(
    "product_category", "region_name", "country"
).agg(
    F.sum("gross_amount").alias("total_gross"),
    F.sum("net_amount").alias("total_net"),
    F.count("product_id").alias("transaction_count")
)

monthly_summary.write.mode("overwrite").saveAsTable("gold.monthly_revenue_summary")
